# Parametric Reduced-Order Model for MRST Reservoir Simulation
## POD + Operator Inference (OpInf) with Affine Parameter Decomposition

---

### Problem Statement

We want a **fast surrogate** that predicts the reservoir pressure field
$p(\mathbf{x}, t; k, S)$ and bottom-hole pressure (BHP) for any
permeability $k$ [mD] and skin factor $S$ without re-running the full
MRST simulator.

---

### Governing Equation

Single-phase, slightly-compressible flow on a 2-D Cartesian grid:

$$\phi\,c_t \,\frac{\partial p}{\partial t}
= \nabla\!\cdot\!\!\left(\frac{k}{\mu}\,\nabla p\right) + q_w(t;k,S)$$

Working with **drawdown** $\delta p = p_{\rm init} - p$ (zero initial
conditions simplifies the ROM to zero ICs).

---

### Affine Parameter Decomposition

The system stiffness matrix decomposes as:

| Symbol | Formula | Physical origin |
|--------|---------|-----------------|
| $\theta_1(k)$ | $k$ | Bulk transmissibility $\propto k$ |
| $\theta_2(k,S)$ | $k\;/\;\bigl(\ln(r_e/r_w)-0.75+S\bigr)$ | Peaceman well index |

---

### Reduced-Order ODE (Operator Inference)

POD approximation: $\delta p \approx \boldsymbol\Phi\,\boldsymbol\alpha(t)$,
$\;\boldsymbol\Phi\in\mathbb{R}^{N_h\times r}$.

After Galerkin projection the **r-dimensional ROM** reads:

$$\dot{\boldsymbol\alpha}
= \bigl[\theta_1\,\hat{\mathbf A}_1 + \theta_2\,\hat{\mathbf A}_2\bigr]
  \boldsymbol\alpha
  + \theta_2\,\hat{\mathbf b}_1 + \hat{\mathbf b}_0
\tag{ROM}$$

Operators $\hat{\mathbf A}_1,\hat{\mathbf A}_2\in\mathbb{R}^{r\times r}$
and $\hat{\mathbf b}_1,\hat{\mathbf b}_0\in\mathbb{R}^r$
are learnt from training data via **least-squares regression**.

---

### BHP Post-Processing (Peaceman formula)

$$\text{WI}\;[\text{m}^3]
= \frac{2\pi\,k\,[\text{m}^2]\,h\,[\text{m}]}{\ln(r_e/r_w)-0.75+S}
\qquad(\mu \text{ NOT in denominator})$$

$$\Delta p\;[\text{Pa}]
= q_{\rm SI}\,[\text{m}^3\text{/s}]\;\times\;\mu_{\rm SI}\;/\;\text{WI}$$

$$p_{\rm BHP}\;[\text{psia}]
= p_{wc}\;[\text{psia}] - \Delta p\;/\;6894.76$$

---

### Workflow

```
Training data (Schedule 1 only: constant rate 800 STB/d)
  (k, S)  ---> stack drawdown snapshots
          ---> randomised SVD --> POD basis Phi  (r modes)
          ---> project --> alpha(t) per case
          ---> OpInf regression --> A1, A2, b1, b0
          ---> ROM: integrate ODE --> alpha_hat(t)
          ---> reconstruct dp approx Phi @ alpha_hat --> BHP

Baseline (POD-I): RBF interpolation of alpha(t) in (log10 k, S) space.
```


## 1 · Imports & Configuration

In [ ]:
# ---- Standard library -------------------------------------------------
import json, pickle, warnings
from pathlib import Path

# ---- Numerical / scientific -------------------------------------------
import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
from scipy.interpolate import RBFInterpolator
from sklearn.utils.extmath import randomized_svd
import h5py

# ---- Plotting ---------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import pandas as pd
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# ---- Paths ------------------------------------------------------------
TRAINING_DIR = Path('../data/01_training')
ROM_DIR      = Path('../rom');     ROM_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR      = Path('../figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- Grid -------------------------------------------------------------
NX, NY   = 50, 50
N_CELLS  = NX * NY          # 2500

# MRST numbers cells x-first (ix fastest).  Centre cell:
#   MATLAB index = ix + (iy-1)*NX = 25 + 24*50 = 1225  (1-based)
#   Python index = (iy-1)*NX + (ix-1) = 24*50 + 24 = 1224
WELL_IDX = 1224

# ---- Physical constants -----------------------------------------------
P_INIT_PSIA = 4500.521193
RE_FT       = 2460.63       # drainage radius [ft]
RW_FT       = 0.350066      # wellbore radius  [ft]
LN_RE_RW    = np.log(RE_FT / RW_FT)   # approx 8.857
H_FT        = 150.0
MU_CP       = 1.2
BO          = 1.15
Q_STBD      = 800.0         # Schedule 1 rate
PHI         = 0.18
CT_PSI1     = 1.2e-5

# ---- SI conversions ---------------------------------------------------
MD_TO_M2  = 9.869233e-16
FT_TO_M   = 0.3048
CP_TO_PAS = 1.0e-3
BBL_TO_M3 = 0.158987
PSI_TO_PA = 6894.76

H_M       = H_FT  * FT_TO_M
MU_SI     = MU_CP * CP_TO_PAS
Q_SI_BASE = Q_STBD * BO * BBL_TO_M3 / 86400.0   # m^3/s reservoir

# ---- ROM hyper-parameters ---------------------------------------------
SCHEDULE_ID   = 1
N_MODES_MAX   = 100
ENERGY_THRESH = 0.9999
LAMBDA_SCALE  = 1e-6

# ---- Train / validation split -----------------------------------------
VAL_K = {70.0, 250.0}
VAL_S = {3.0,  7.0}

print("Configuration loaded.")
print(f"  ln(re/rw) = {LN_RE_RW:.4f}")
print(f"  Q_SI_base = {Q_SI_BASE:.4e} m^3/s")


## 2 · Data Loading Utilities

Each simulation case lives in a folder named
`k{k_mD:03d}_s{S:02d}_sched{id}` and contains:

| File | Shape | Notes |
|------|-------|-------|
| `snapshots_psia.mat` (MATLAB v7.3 / HDF5) | `[2500 x Nt]` | Pressure field |
| `well_data.mat` (HDF5) | vectors | BHP, rate, time, dp |
| `params.json` | scalars | All parameters |

> **Transpose note:** `h5py` reads MATLAB v7.3 matrices in C-order, so
> a `[2500 x Nt]` MATLAB matrix arrives as `(Nt, 2500)`.
> We call `.T` after reading to restore `(N_cells, Nt)`.


In [ ]:
def load_snapshots(case_dir):
    """Load drawdown dp = p_init - p as (N_cells, Nt) float32."""
    with h5py.File(case_dir / "snapshots_psia.mat", "r") as f:
        X = f["X_psia"][:].T.astype(np.float32)   # (N_cells, Nt)
    return P_INIT_PSIA - X


def load_well_data(case_dir):
    """Return dict with time_hr, bhp_psia, rate_STBd, delta_p_psi."""
    out = {}
    with h5py.File(case_dir / "well_data.mat", "r") as f:
        for key in ("time_hr", "bhp_psia", "rate_STBd", "delta_p_psi"):
            out[key] = f[key][:].flatten()
    return out


def load_params(case_dir):
    with open(case_dir / "params.json") as fh:
        return json.load(fh)


def load_case(case_dir):
    """Load one simulation case completely."""
    params = load_params(case_dir)
    dp     = load_snapshots(case_dir)
    well   = load_well_data(case_dir)
    return dict(
        name    = case_dir.name,
        k_mD    = float(params["k_mD"]),
        S       = float(params["skin"]),
        sched   = int(params.get("schedule_id", SCHEDULE_ID)),
        dp      = dp,
        time_hr = well["time_hr"],
        bhp     = well["bhp_psia"],
        rate    = well["rate_STBd"],
    )


def scan_schedule(training_dir, schedule_id=1):
    """Return list of metadata dicts for all cases with given schedule."""
    records = []
    for p in sorted(training_dir.iterdir()):
        if not p.is_dir():
            continue
        pjson = p / "params.json"
        if not pjson.exists():
            continue
        params = load_params(p)
        if int(params.get("schedule_id", -1)) != schedule_id:
            continue
        records.append(dict(
            path = p,
            k_mD = float(params["k_mD"]),
            S    = float(params["skin"]),
        ))
    return records


print("Data loading utilities defined.")


## 3 · Scan Training Directory & Train / Validation Split

We restrict to **Schedule 1** (constant rate 800 STB/d) — the simplest
dynamics where the affine decomposition is exact.

**Held-out parameters (interpolation test):**
- $k_{\rm val} \in \{70,\,250\}$ mD
- $S_{\rm val} \in \{3,\,7\}$

| Set | k values | S values | Count |
|-----|----------|----------|-------|
| Training | 10,15,20,30,50,100,150,200,300 (9 values) | 0,1,2,4,5,6,8 (7 values) | 63 |
| Validation | includes k in {70,250} OR S in {3,7} | — | 36 |


In [ ]:
all_cases = scan_schedule(TRAINING_DIR, schedule_id=SCHEDULE_ID)
print(f"Total Schedule-{SCHEDULE_ID} cases found: {len(all_cases)}")

train_meta, val_meta = [], []
for rec in all_cases:
    is_val = (rec["k_mD"] in VAL_K) or (rec["S"] in VAL_S)
    (val_meta if is_val else train_meta).append(rec)

print(f"  Training:   {len(train_meta)} cases")
print(f"  Validation: {len(val_meta)}  cases")

# Load training snapshots
print("\nLoading training snapshots ...")
train_cases = []
for rec in train_meta:
    c = load_case(rec["path"])
    train_cases.append(c)
    print(f"  {c['name']:30s}  k={c['k_mD']:6.1f} mD  S={c['S']:.0f}"
          f"  shape={c['dp'].shape}", flush=True)

Nt      = train_cases[0]["dp"].shape[1]
time_hr = train_cases[0]["time_hr"]
assert all(c["dp"].shape[1] == Nt for c in train_cases), "Nt mismatch!"
print(f"\nNt = {Nt},  t in [{time_hr[0]:.4f}, {time_hr[-1]:.1f}] hr")

print("\nLoading validation cases ...")
val_cases = [load_case(r["path"]) for r in val_meta]
print(f"Loaded {len(val_cases)} validation cases.")


## 4 · Exploratory Data Analysis

Visualise the pressure drawdown structure and BHP behaviour across the
parameter space before building the ROM.


In [ ]:
def reshape_field(v):
    """(N_cells,) -> (NY, NX), MRST Fortran (x-first) layout."""
    return v.reshape(NY, NX, order="F")


low_k  = min(train_cases, key=lambda c: c["k_mD"])
high_k = max(train_cases, key=lambda c: c["k_mD"])

t_idx    = [0, Nt//4, Nt//2, -1]
t_labels = [f"t={time_hr[i]:.1f} hr" for i in t_idx]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, case in enumerate([low_k, high_k]):
    for col, ti in enumerate(t_idx):
        ax = axes[row, col]
        field = reshape_field(case["dp"][:, ti])
        im = ax.imshow(field, origin="lower", cmap="hot",
                       vmin=0, vmax=case["dp"].max())
        ax.set_title(t_labels[col], fontsize=8)
        if col == 0:
            ax.set_ylabel(f"k={case['k_mD']:.0f} mD  S={case['S']:.0f}",
                          fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.046, label="dp [psi]")
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Drawdown snapshots - low k (top) vs high k (bottom)")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_snapshots.png", bbox_inches="tight")
plt.show()


In [ ]:
# BHP curves: effect of skin at fixed k=100 mD
cases_k100 = sorted([c for c in train_cases if c["k_mD"] == 100.0],
                    key=lambda c: c["S"])
S_vals = [c["S"] for c in cases_k100]
norm   = mcolors.Normalize(vmin=min(S_vals), vmax=max(S_vals))
cmap   = plt.cm.viridis

fig, ax = plt.subplots(figsize=(8, 4))
for c in cases_k100:
    ax.semilogx(c["time_hr"], c["bhp"],
                color=cmap(norm(c["S"])), lw=1.5, label=f"S={c['S']:.0f}")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label="Skin factor S")
ax.set_xlabel("Time [hr]"); ax.set_ylabel("BHP [psia]")
ax.set_title("BHP at k=100 mD - effect of skin factor")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_bhp_skin.png", bbox_inches="tight")
plt.show()


## 5 · POD Basis Construction (Randomised SVD)

Stack **all training drawdown snapshots** into one matrix
$\mathbf D \in \mathbb{R}^{N_h \times (N_{\rm train}\cdot N_t)}$ and compute
its truncated SVD:

$$\mathbf D = \mathbf U\,\boldsymbol\Sigma\,\mathbf V^\top$$

The POD basis is $\boldsymbol\Phi = \mathbf U[:, 1{:}r]$ where

$$\frac{\sum_{i=1}^r \sigma_i^2}{\sum_{i=1}^{N_{\rm modes}} \sigma_i^2}
\geq 0.9999$$

We use `sklearn.utils.extmath.randomized_svd` which avoids forming the full
$N_h \times N_h$ covariance matrix.


In [ ]:
print("Stacking training snapshots ...")
D = np.hstack([c["dp"] for c in train_cases])   # (N_cells, N_train*Nt)
print(f"  Snapshot matrix D: {D.shape}  ({D.nbytes/1e6:.0f} MB)")

print(f"Running randomised SVD (n_components={N_MODES_MAX}) ...")
U, sigma, Vt = randomized_svd(D.astype(np.float64),
                               n_components=N_MODES_MAX,
                               random_state=42)
print(f"  sigma range: [{sigma[-1]:.2e}, {sigma[0]:.2e}]")

energy     = sigma**2
cum_energy = np.cumsum(energy) / energy.sum()

r = int(np.searchsorted(cum_energy, ENERGY_THRESH)) + 1
print(f"\n  Modes for {ENERGY_THRESH*100:.2f}% energy: r = {r}")
for thresh in (0.99, 0.999, 0.9999):
    ri = int(np.searchsorted(cum_energy, thresh)) + 1
    print(f"  {thresh*100:.2f}% -> r = {ri}")

Phi     = U[:, :r]
sigma_r = sigma[:r]

# ---- Plot ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.semilogy(range(1, len(sigma)+1), sigma, "o-", ms=3)
ax.axvline(r, color="r", ls="--", label=f"r={r}")
ax.set_xlabel("Mode index"); ax.set_ylabel("Singular value")
ax.set_title("Singular value spectrum"); ax.legend(); ax.grid(True, alpha=.3)

ax = axes[1]
ax.plot(range(1, len(cum_energy)+1), cum_energy*100, "b-", lw=1.5)
ax.axhline(ENERGY_THRESH*100, color="r", ls="--",
           label=f"{ENERGY_THRESH*100:.2f}%")
ax.axvline(r, color="r", ls="--")
ax.set_xlabel("Mode index"); ax.set_ylabel("Cumulative energy [%]")
ax.set_title("Cumulative energy"); ax.legend(); ax.grid(True, alpha=.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_singular_values.png", bbox_inches="tight")
plt.show()
print(f"\nPOD basis Phi: {Phi.shape}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    if i >= r: ax.axis("off"); continue
    mode2d = reshape_field(Phi[:, i])
    vmax   = np.abs(mode2d).max()
    ax.imshow(mode2d, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Mode {i+1}  (sigma={sigma_r[i]:.2e})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("First 6 POD modes (drawdown, psi)", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_pod_modes.png", bbox_inches="tight")
plt.show()


## 6 · Reduced Coordinates

Project each case's drawdown trajectory onto the POD basis:

$$\boldsymbol\alpha(t;\,k,S)
= \boldsymbol\Phi^\top \,\delta p(t;\,k,S)
\in \mathbb{R}^r$$

These $r$-dimensional trajectories are the data the Operator Inference
step will fit.


In [ ]:
# Project all cases
for c in train_cases:
    c["alpha"] = Phi.T @ c["dp"].astype(np.float64)   # (r, Nt)
for c in val_cases:
    c["alpha"] = Phi.T @ c["dp"].astype(np.float64)

# Sanity check: reconstruction error
c0 = train_cases[0]
dp_r     = Phi @ c0["alpha"]
rel_err  = la.norm(c0["dp"].astype(np.float64) - dp_r, "fro")          / la.norm(c0["dp"].astype(np.float64), "fro")
print(f"POD reconstruction error (case 0): {rel_err:.2e}")

# Visualise modal trajectories coloured by k
k_vals = sorted(set(c["k_mD"] for c in train_cases))
norm_k = mcolors.LogNorm(vmin=min(k_vals), vmax=max(k_vals))
cmap   = plt.cm.plasma

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")
for i, ax in enumerate(axes.flat):
    if i >= r: ax.axis("off"); continue
    for c in train_cases:
        ax.semilogx(time_hr, c["alpha"][i],
                    color=cmap(norm_k(c["k_mD"])), lw=0.8, alpha=0.7)
    ax.set_title(f"Mode {i+1}", fontsize=9)
    ax.set_xlabel("t [hr]", fontsize=8)
    ax.set_ylabel(f"alpha_{i+1} [psi]", fontsize=8)
    ax.grid(True, which="both", alpha=0.2)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_k)
fig.colorbar(sm, ax=axes.ravel().tolist(), label="k [mD]", fraction=0.02)
fig.suptitle("Reduced coordinates alpha(t) for all training cases", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_reduced_coords.png", bbox_inches="tight")
plt.show()


## 7 · Operator Inference (OpInf)

### Affine decomposition

The reduced ODE is:

$$\dot{\boldsymbol\alpha}
= \underbrace{\theta_1}_{=k} \hat{\mathbf A}_1\boldsymbol\alpha
+ \underbrace{\theta_2}_{=\text{WI}} \hat{\mathbf A}_2\boldsymbol\alpha
+ \theta_2\,\hat{\mathbf b}_1
+ \hat{\mathbf b}_0$$

For each training case $(k_i, S_i)$ at time step $t_j$ the **feature vector** is:

$$\mathbf f_{ij}
= \bigl[\theta_1^{(i)}\boldsymbol\alpha_{ij}^\top,\;
         \theta_2^{(i)}\boldsymbol\alpha_{ij}^\top,\;
         \theta_2^{(i)},\;
         1\bigr]^\top \in \mathbb{R}^{2r+2}$$

Stacking over all cases and time steps gives a linear system

$$\mathbf F\,\mathbf O^\top = \dot{\mathbf A}$$

solved by Tikhonov-regularised least squares.

### Time derivatives

`np.gradient` (second-order central differences, non-uniform time spacing)
is used to estimate $\dot{\boldsymbol\alpha}$ from snapshots.


In [ ]:
def theta_params(k, S):
    """Return (theta1, theta2) affine parameter functions."""
    th1 = k                                   # mD
    th2 = k / (LN_RE_RW - 0.75 + S)         # mD / dimensionless
    return th1, th2


def time_deriv(alpha, t):
    """Estimate d(alpha)/dt via np.gradient (non-uniform spacing).
    alpha: (r, Nt),  t: (Nt,)  -> returns (r, Nt)
    """
    dalpha = np.zeros_like(alpha)
    for i in range(alpha.shape[0]):
        dalpha[i] = np.gradient(alpha[i], t)
    return dalpha


print("Building OpInf regression dataset ...")
F_list  = []
dA_list = []

for c in train_cases:
    th1, th2 = theta_params(c["k_mD"], c["S"])
    alpha    = c["alpha"]                    # (r, Nt)
    dalpha   = time_deriv(alpha, time_hr)   # (r, Nt)
    feat = np.vstack([
        th1 * alpha,
        th2 * alpha,
        th2 * np.ones((1, Nt)),
        np.ones((1, Nt)),
    ])                                       # (2r+2, Nt)
    F_list.append(feat.T)                   # (Nt, 2r+2)
    dA_list.append(dalpha.T)               # (Nt, r)

F  = np.vstack(F_list)
dA = np.vstack(dA_list)
print(f"  Feature matrix F:  {F.shape}")
print(f"  Target matrix dA:  {dA.shape}")

lam    = LAMBDA_SCALE * (la.norm(F, "fro")**2 / F.shape[0])
F_reg  = np.vstack([F,  lam * np.eye(F.shape[1])])
dA_reg = np.vstack([dA, np.zeros((F.shape[1], dA.shape[1]))])
print(f"  lambda = {lam:.3e}  (Tikhonov)")

print("Solving least-squares system ...")
O, res, rank, sv = la.lstsq(F_reg, dA_reg, check_finite=False)
O = O.T    # (r, 2r+2)

A1_hat = O[:, :r]
A2_hat = O[:, r:2*r]
b1_hat = O[:, 2*r]
b0_hat = O[:, 2*r+1]

print(f"\nOperator norms:")
print(f"  ||A1|| = {la.norm(A1_hat):.3e}  (bulk diffusion)")
print(f"  ||A2|| = {la.norm(A2_hat):.3e}  (well coupling)")
print(f"  ||b1|| = {la.norm(b1_hat):.3e}  (well forcing)")
print(f"  ||b0|| = {la.norm(b0_hat):.3e}  (constant offset)")


In [ ]:
# ---- Residual check ---------------------------------------------------
dA_pred = F @ O.T
rel_res  = la.norm(dA - dA_pred, "fro") / la.norm(dA, "fro")
print(f"OpInf training residual: {rel_res:.3e}")

# Plot dα/dt vs fitted for first 4 modes on one training case
c0 = train_cases[0]
th1, th2 = theta_params(c0["k_mD"], c0["S"])
feat0    = np.vstack([th1*c0["alpha"], th2*c0["alpha"],
                      th2*np.ones((1,Nt)), np.ones((1,Nt))]).T
dA0_pred = feat0 @ O.T

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for i, ax in enumerate(axes.flat):
    if i >= r: break
    ax.semilogx(time_hr, time_deriv(c0["alpha"], time_hr)[i],
                "k-", lw=1.5, label="FOM d(alpha)/dt")
    ax.semilogx(time_hr, dA0_pred[:, i], "r--", lw=1.5, label="OpInf")
    ax.set_title(f"Mode {i+1}", fontsize=9)
    ax.set_xlabel("t [hr]", fontsize=8)
    ax.legend(fontsize=7); ax.grid(True, alpha=.3)
fig.suptitle(f"OpInf fit - case {c0['name']}", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_opinf_residual.png", bbox_inches="tight")
plt.show()


## 8 · ROM Prediction Function

Given new parameters $(k^*, S^*)$:

1. $\theta_1^* = k^*$, $\quad\theta_2^* = k^*\!/(\ln(r_e/r_w)-0.75+S^*)$
2. $\mathbf A^* = \theta_1^*\hat{\mathbf A}_1 + \theta_2^*\hat{\mathbf A}_2$, $\quad\mathbf b^* = \theta_2^*\hat{\mathbf b}_1 + \hat{\mathbf b}_0$
3. Integrate $\dot{\boldsymbol\alpha} = \mathbf A^*\boldsymbol\alpha + \mathbf b^*$ from $\boldsymbol\alpha(0)=\mathbf 0$ using `solve_ivp(method='Radau')`
4. Reconstruct $\delta p \approx \boldsymbol\Phi\boldsymbol\alpha(t)$
5. Compute BHP from well-cell pressure via Peaceman formula

> **Unit note:** $\mu$ is **not** in the WI denominator — it enters the
> pressure-drop equation separately.


In [ ]:
def bhp_from_field(dp_full, k_mD, S):
    """Compute BHP [psia] from drawdown field (N_cells, Nt)."""
    k_SI  = k_mD * MD_TO_M2
    WI_SI = 2 * np.pi * k_SI * H_M / (LN_RE_RW - 0.75 + S)  # m^3
    dP_Pa = Q_SI_BASE * MU_SI / WI_SI                         # Pa
    p_wc  = P_INIT_PSIA - dp_full[WELL_IDX, :]               # psia
    return p_wc - dP_Pa / PSI_TO_PA


def predict_rom(k_new, S_new, t_eval, Phi, A1, A2, b1, b0):
    """
    Predict drawdown field and BHP for new (k, S).
    Returns dict: dp_rom (N_cells, Nt), bhp_rom (Nt,), alpha (r, Nt).
    """
    th1, th2 = theta_params(k_new, S_new)
    A_star   = th1 * A1 + th2 * A2
    b_star   = th2 * b1 + b0

    def ode(t, a):
        return A_star @ a + b_star

    sol = solve_ivp(
        ode,
        t_span = (t_eval[0], t_eval[-1]),
        y0     = np.zeros(A_star.shape[0]),
        method = "Radau",
        t_eval = t_eval,
        rtol   = 1e-7,
        atol   = 1e-9,
    )
    alpha_hat = sol.y                        # (r, Nt)
    dp_rom    = Phi @ alpha_hat             # (N_cells, Nt)
    bhp_rom   = bhp_from_field(dp_rom, k_new, S_new)
    return dict(dp_rom=dp_rom, bhp_rom=bhp_rom, alpha=alpha_hat)


print("ROM prediction function defined.")

# Quick test on first training case
c0   = train_cases[0]
pred = predict_rom(c0["k_mD"], c0["S"], time_hr,
                   Phi, A1_hat, A2_hat, b1_hat, b0_hat)
print(f"Test prediction shape: dp={pred['dp_rom'].shape}, bhp={pred['bhp_rom'].shape}")


## 9 · Training Reconstruction Check

Verify that the ROM accurately reproduces a training case.


In [ ]:
# Pick a mid-range training case
c_check = next((c for c in train_cases
                if c["k_mD"] == 100.0 and c["S"] == 4.0),
               train_cases[len(train_cases)//2])

print(f"Checking case: {c_check['name']}")
pred = predict_rom(c_check["k_mD"], c_check["S"], time_hr,
                   Phi, A1_hat, A2_hat, b1_hat, b0_hat)

dp_fom      = c_check["dp"].astype(np.float64)
bhp_fom_chk = bhp_from_field(dp_fom, c_check["k_mD"], c_check["S"])
err_dp  = la.norm(dp_fom - pred["dp_rom"], "fro") / la.norm(dp_fom, "fro")
err_bhp = np.linalg.norm(bhp_fom_chk - pred["bhp_rom"])         / np.linalg.norm(bhp_fom_chk)
print(f"  Relative ||dp||  error: {err_dp:.2e}")
print(f"  Relative ||BHP|| error: {err_bhp:.2e}")

fig = plt.figure(figsize=(13, 5))
gs  = GridSpec(1, 3, figure=fig, wspace=0.4)

ax0 = fig.add_subplot(gs[0])
ax0.semilogx(time_hr, c_check["bhp"],     "k-",  lw=2,   label="FOM")
ax0.semilogx(time_hr, pred["bhp_rom"],    "r--", lw=1.5, label="ROM")
ax0.set_xlabel("t [hr]"); ax0.set_ylabel("BHP [psia]")
ax0.set_title(f"BHP  k={c_check['k_mD']:.0f} mD  S={c_check['S']:.0f}")
ax0.legend(); ax0.grid(True, alpha=.3)

t_mid = Nt // 2
for col_idx, (gs_id, title, data) in enumerate([
    (gs[1], "FOM drawdown", reshape_field(dp_fom[:, t_mid])),
    (gs[2], "ROM error x1000",
     reshape_field((dp_fom[:, t_mid] - pred["dp_rom"][:, t_mid]) * 1e3)),
]):
    ax = fig.add_subplot(gs_id)
    cmap_use = "hot" if col_idx == 0 else "RdBu_r"
    vmax = np.abs(data).max()
    im   = ax.imshow(data, origin="lower", cmap=cmap_use,
                     vmin=(0 if col_idx == 0 else -vmax),
                     vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f"{title}  t={time_hr[t_mid]:.1f} hr", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Training reconstruction check", fontsize=11)
plt.savefig(FIG_DIR / "07_train_check.png", bbox_inches="tight")
plt.show()


## 10 · Validation on Held-Out Cases

Error metrics:

$$\varepsilon_p
= \frac{\|\delta p_{\rm ROM} - \delta p_{\rm FOM}\|_F}
       {\|\delta p_{\rm FOM}\|_F}, \qquad
\varepsilon_{\rm BHP}
= \frac{\|p^{\rm BHP}_{\rm ROM} - p^{\rm BHP}_{\rm FOM}\|_2}
       {\|p^{\rm BHP}_{\rm FOM}\|_2}$$


In [ ]:
print("Validating OpInf ROM on held-out cases ...")
opinf_results = []

for c in val_cases:
    pred      = predict_rom(c["k_mD"], c["S"], time_hr,
                            Phi, A1_hat, A2_hat, b1_hat, b0_hat)
    dp_fom    = c["dp"].astype(np.float64)
    bhp_fom_v = bhp_from_field(dp_fom, c["k_mD"], c["S"])

    eps_p   = la.norm(dp_fom - pred["dp_rom"], "fro") / la.norm(dp_fom, "fro")
    eps_bhp = np.linalg.norm(bhp_fom_v - pred["bhp_rom"])             / np.linalg.norm(bhp_fom_v)

    opinf_results.append(dict(
        name     = c["name"],
        k_mD     = c["k_mD"],
        S        = c["S"],
        eps_p    = eps_p,
        eps_bhp  = eps_bhp,
        pred_bhp = pred["bhp_rom"],
        fom_bhp  = bhp_fom_v,
    ))
    print(f"  {c['name']:28s}  eps_p={eps_p:.2e}  eps_BHP={eps_bhp:.2e}")

df_val = pd.DataFrame([{k: v for k, v in r.items()
                         if k not in ("pred_bhp", "fom_bhp")}
                        for r in opinf_results])
print(f"\nMean eps_p   = {df_val['eps_p'].mean():.3e}")
print(f"Mean eps_BHP = {df_val['eps_bhp'].mean():.3e}")
print(df_val.sort_values("eps_p").to_string(index=False))


In [ ]:
# ---- Error heatmap ----------------------------------------------------
k_val_arr = sorted(df_val["k_mD"].unique())
S_val_arr = sorted(df_val["S"].unique())
eps_grid  = np.full((len(S_val_arr), len(k_val_arr)), np.nan)

for row in opinf_results:
    ki = k_val_arr.index(row["k_mD"])
    si = S_val_arr.index(row["S"])
    eps_grid[si, ki] = row["eps_p"] * 100

fig, ax = plt.subplots(figsize=(max(6, len(k_val_arr)*0.8),
                                 max(4, len(S_val_arr)*0.6)))
im = ax.imshow(eps_grid, aspect="auto", cmap="YlOrRd", origin="lower")
ax.set_xticks(range(len(k_val_arr)));  ax.set_xticklabels(k_val_arr, rotation=45)
ax.set_yticks(range(len(S_val_arr)));  ax.set_yticklabels(S_val_arr)
ax.set_xlabel("k [mD]"); ax.set_ylabel("S")
ax.set_title("eps_p [%] on validation cases (OpInf ROM)")
plt.colorbar(im, ax=ax, label="Relative eps_p [%]")
for si in range(len(S_val_arr)):
    for ki in range(len(k_val_arr)):
        if not np.isnan(eps_grid[si, ki]):
            ax.text(ki, si, f"{eps_grid[si,ki]:.2f}",
                    ha="center", va="center", fontsize=7,
                    color="black" if eps_grid[si,ki] < 5 else "white")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_val_heatmap.png", bbox_inches="tight")
plt.show()


In [ ]:
# ---- BHP plots (3 best + 3 worst) ------------------------------------
sample = sorted(opinf_results, key=lambda x: x["eps_bhp"])
sample = sample[:3] + sample[-3:]

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
for ax, row in zip(axes.flat, sample):
    ax.semilogx(time_hr, row["fom_bhp"], "k-",  lw=2,   label="FOM")
    ax.semilogx(time_hr, row["pred_bhp"], "r--", lw=1.5, label="ROM")
    ax.set_title(f"k={row['k_mD']:.0f} mD  S={row['S']:.0f}\n"
                 f"eps_BHP={row['eps_bhp']:.2e}", fontsize=8)
    ax.set_xlabel("t [hr]", fontsize=8)
    ax.set_ylabel("BHP [psia]", fontsize=8)
    ax.legend(fontsize=7); ax.grid(True, alpha=.3)
fig.suptitle("Validation BHP - OpInf ROM (3 best + 3 worst)", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "09_val_bhp.png", bbox_inches="tight")
plt.show()


## 11 · Baseline: POD with RBF Interpolation (POD-I)

For **each time step** $t_j$ we could fit a separate RBF, but it is more
efficient to fit **one vector-valued RBF** that maps

$$(\log_{10}k,\,S) \;\longrightarrow\;
  [\alpha_1(t_1),\ldots,\alpha_r(t_{N_t})] \in \mathbb{R}^{r\cdot N_t}$$

over all training parameter pairs simultaneously.
At inference we reshape the output back to $(r, N_t)$ and reconstruct
$\delta p = \boldsymbol\Phi\hat{\boldsymbol\alpha}$.

POD-I requires **no ODE integration** — pure regression in parameter space.


In [ ]:
print("Building POD-I RBF interpolant ...")

# Training points in (log10 k, S) space
X_train = np.array([[np.log10(c["k_mD"]), c["S"]] for c in train_cases])

# Stack alpha(t) per case: shape per case is (r, Nt)
# We want A_train[i] = alpha(t) flattened row-major from (Nt, r)
# so that reshape(Nt, r) later recovers (Nt, r) -> .T = (r, Nt)
A_train = np.array([c["alpha"].T.flatten() for c in train_cases])
# A_train: (Nc, Nt*r)

rbf_interp = RBFInterpolator(
    X_train, A_train,
    kernel="thin_plate_spline",
    smoothing=0.0,
)
print(f"  RBF fitted on {len(train_cases)} training parameter pairs.")


def predict_pod_i(k_new, S_new):
    """Predict using POD-I."""
    x_q       = np.array([[np.log10(k_new), S_new]])
    a_vec     = rbf_interp(x_q)[0]              # (Nt*r,)
    alpha_hat = a_vec.reshape(Nt, r).T          # (r, Nt)
    dp_podi   = Phi @ alpha_hat
    bhp_podi  = bhp_from_field(dp_podi, k_new, S_new)
    return dict(dp_podi=dp_podi, bhp_podi=bhp_podi, alpha=alpha_hat)


print("Validating POD-I ...")
podi_results = []
for c in val_cases:
    pred    = predict_pod_i(c["k_mD"], c["S"])
    dp_fom  = c["dp"].astype(np.float64)
    bhp_fom_v = bhp_from_field(dp_fom, c["k_mD"], c["S"])
    eps_p   = la.norm(dp_fom - pred["dp_podi"], "fro") / la.norm(dp_fom, "fro")
    eps_bhp = np.linalg.norm(bhp_fom_v - pred["bhp_podi"])             / np.linalg.norm(bhp_fom_v)
    podi_results.append(dict(name=c["name"], k_mD=c["k_mD"], S=c["S"],
                              eps_p=eps_p, eps_bhp=eps_bhp))

df_podi = pd.DataFrame(podi_results)
print(f"\nPOD-I  Mean eps_p   = {df_podi['eps_p'].mean():.3e}")
print(f"POD-I  Mean eps_BHP = {df_podi['eps_bhp'].mean():.3e}")


## 12 · Method Comparison: OpInf vs POD-I

In [ ]:
df_compare = pd.DataFrame({
    "k_mD"         : df_val["k_mD"].values,
    "S"            : df_val["S"].values,
    "OpInf eps_p"  : df_val["eps_p"].values,
    "POD-I eps_p"  : df_podi["eps_p"].values,
    "OpInf eps_BHP": df_val["eps_bhp"].values,
    "POD-I eps_BHP": df_podi["eps_bhp"].values,
})
print(df_compare.to_string(index=False, float_format="{:.3e}".format))

methods  = ["OpInf", "POD-I"]
mean_ep  = [df_val["eps_p"].mean(),   df_podi["eps_p"].mean()]
mean_bhp = [df_val["eps_bhp"].mean(), df_podi["eps_bhp"].mean()]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, vals, title in zip(axes,
                            [mean_ep, mean_bhp],
                            ["Mean relative eps_p", "Mean relative eps_BHP"]):
    bars = ax.bar(methods, vals, color=["steelblue", "tomato"], width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height()*1.02,
                f"{val:.2e}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Relative error"); ax.set_title(title)
    ax.grid(True, axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "10_method_comparison.png", bbox_inches="tight")
plt.show()


## 13 · Save ROM Artefacts

| File | Contents |
|------|---------|
| `rom_operators.npz` | `Phi`, `sigma`, `A1`, `A2`, `b1`, `b0`, `time_hr` |
| `pod_i_interps.pkl` | The fitted `RBFInterpolator` |
| `rom_metadata.json` | Configuration, split info, error summary |


In [ ]:
np.savez_compressed(
    ROM_DIR / "rom_operators.npz",
    Phi=Phi, sigma=sigma_r,
    A1=A1_hat, A2=A2_hat,
    b1=b1_hat, b0=b0_hat,
    time_hr=time_hr,
)
print(f"Saved  {ROM_DIR / 'rom_operators.npz'}")

with open(ROM_DIR / "pod_i_interps.pkl", "wb") as fh:
    pickle.dump(rbf_interp, fh)
print(f"Saved  {ROM_DIR / 'pod_i_interps.pkl'}")

metadata = dict(
    schedule           = SCHEDULE_ID,
    N_cells            = N_CELLS,
    NX=NX, NY=NY,
    WELL_IDX           = WELL_IDX,
    r                  = int(r),
    energy_captured    = float(cum_energy[r-1]),
    energy_thresh      = ENERGY_THRESH,
    Nt                 = int(Nt),
    n_train            = len(train_cases),
    n_val              = len(val_cases),
    val_K              = sorted(VAL_K),
    val_S              = sorted(VAL_S),
    ln_re_rw           = float(LN_RE_RW),
    p_init_psia        = P_INIT_PSIA,
    mu_cp=MU_CP, Bo=BO, Q_STBD=Q_STBD,
    h_ft=H_FT, re_ft=RE_FT, rw_ft=RW_FT,
    opinf_mean_eps_p   = float(df_val["eps_p"].mean()),
    opinf_mean_eps_bhp = float(df_val["eps_bhp"].mean()),
    podi_mean_eps_p    = float(df_podi["eps_p"].mean()),
    podi_mean_eps_bhp  = float(df_podi["eps_bhp"].mean()),
)
with open(ROM_DIR / "rom_metadata.json", "w") as fh:
    json.dump(metadata, fh, indent=2)
print(f"Saved  {ROM_DIR / 'rom_metadata.json'}")

print("\n" + "="*55)
print("ROM Summary")
print("="*55)
print(f"  POD rank r          : {r}")
print(f"  Energy captured     : {cum_energy[r-1]*100:.4f}%")
print(f"  Training cases      : {len(train_cases)}")
print(f"  Validation cases    : {len(val_cases)}")
print(f"  OpInf  eps_p (mean) : {df_val['eps_p'].mean():.3e}")
print(f"  OpInf  eps_BHP(mean): {df_val['eps_bhp'].mean():.3e}")
print(f"  POD-I  eps_p (mean) : {df_podi['eps_p'].mean():.3e}")
print(f"  POD-I  eps_BHP(mean): {df_podi['eps_bhp'].mean():.3e}")
print("="*55)
